In [0]:

from pyspark.sql import functions as F

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

# Read Supplier Bronze Parquet
df_supplier = spark.read.parquet(
    f"{BRONZE}/supplier_price_list"
)

# Check schema
df_supplier.printSchema()

# Total rows and exact duplicates
total_rows = df_supplier.count()
unique_rows = df_supplier.distinct().count()

print("Total rows:", total_rows)
print("Exact duplicate rows:", total_rows - unique_rows)

# Inspect supplier prices and date formats
display(df_supplier.limit(20))


root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- supplier_cost: string (nullable = true)
 |-- effective_date: string (nullable = true)

Total rows: 118
Exact duplicate rows: 4


product_id,product_name,supplier_name,supplier_cost,effective_date
1,Wireless Earbuds,Prime Distributors,1107.42,2024-11-09
1,Wireless Earbuds,Prime Distributors,1131.27,2025-06-26
1,Wireless Earbuds,Prime Distributors,1128.97,2026-01-23
2,Bluetooth Speaker,Everest Traders,2059.35,2026-01-18
3,Power Bank 20000mAh,Prime Distributors,"₹1,298.26",2025-10-27
3,Power Bank 20000mAh,Prime Distributors,"₹1,298.26",2025-10-27
3,Power Bank 20000mAh,Prime Distributors,Rs. 1202.45,2026-02-01
3,Power Bank 20000mAh,Prime Distributors,1214.17,2026-06-29
4,USB-C Charger 65W,Everest Traders,Rs. 718.47,2026-05-12
5,Smartwatch,Lotus Wholesale,2951.79,2025-06-08


In [0]:

from pyspark.sql import functions as F

# Step 1: Remove exact duplicate rows
supplier_dedup = df_supplier.dropDuplicates()

# Step 2: Extract numeric cost
cost_text = F.regexp_replace(
    F.regexp_extract(
        F.col("supplier_cost"),
        r"-?[0-9][0-9,]*(\.[0-9]+)?",
        0
    ),
    ",",
    ""
)

# Step 3: Clean columns and parse dates
supplier_parsed = (
    supplier_dedup
    .withColumn(
        "product_id",
        F.expr("try_cast(trim(product_id) AS INT)")
    )
    .withColumn(
        "product_name",
        F.regexp_replace(F.trim("product_name"), r"\s+", " ")
    )
    .withColumn(
        "supplier_name",
        F.regexp_replace(F.trim("supplier_name"), r"\s+", " ")
    )
    .withColumn(
        "supplier_cost",
        F.when(
            cost_text != "",
            cost_text.cast("decimal(12,2)")
        )
    )
    .withColumn(
        "effective_date",
        F.coalesce(
            *[
                F.to_date(
                    F.try_to_timestamp(
                        F.col("effective_date"),
                        F.lit(fmt)
                    )
                )
                for fmt in [
                    "yyyy-MM-dd",
                    "dd/MM/yyyy",
                    "dd-MMM-yyyy"
                ]
            ]
        )
    )
)

# Step 4: Keep only known products
known_products = (
    spark.table("atliq_dl.silver.products")
    .select("product_id")
    .distinct()
)

unknown_count = supplier_parsed.join(
    known_products, "product_id", "left_anti"
).count()

supplier_clean = supplier_parsed.join(
    known_products, "product_id", "left_semi"
)

print("Raw rows:", df_supplier.count())
print("Exact duplicates removed:",
      df_supplier.count() - supplier_dedup.count())
print("Unknown product rows removed:", unknown_count)
print("Cleaned rows:", supplier_clean.count())

display(supplier_clean.orderBy("product_id", "effective_date"))


Raw rows: 118
Exact duplicates removed: 4
Unknown product rows removed: 3
Cleaned rows: 111


product_id,product_name,supplier_name,supplier_cost,effective_date
1,Wireless Earbuds,Prime Distributors,1107.42,2024-11-09
1,Wireless Earbuds,Prime Distributors,1131.27,2025-06-26
1,Wireless Earbuds,Prime Distributors,1128.97,2026-01-23
2,Bluetooth Speaker,Everest Traders,2059.35,2026-01-18
3,Power Bank 20000mAh,Prime Distributors,1298.26,2025-10-27
3,Power Bank 20000mAh,Prime Distributors,1202.45,2026-02-01
3,Power Bank 20000mAh,Prime Distributors,1214.17,2026-06-29
4,USB-C Charger 65W,Everest Traders,718.47,2026-05-12
5,Smartwatch,Lotus Wholesale,2951.79,2025-06-08
5,Smartwatch,Lotus Wholesale,2905.43,2025-11-12


In [0]:

# Check invalid values before writing
invalid_rows = supplier_clean.filter(
    F.col("product_id").isNull() |
    F.col("supplier_name").isNull() |
    (F.col("supplier_name") == "") |
    F.col("supplier_cost").isNull() |
    (F.col("supplier_cost") <= 0) |
    F.col("effective_date").isNull()
).count()

print("Invalid supplier rows:", invalid_rows)

assert invalid_rows == 0, "Invalid supplier records found"

# Full overwrite
TARGET = "atliq_dl.silver.supplier_price_list"

supplier_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET)

print("Supplier Silver table created!")

display(
    spark.table(TARGET).agg(
        F.count("*").alias("silver_rows"),
        F.countDistinct("product_id").alias("unique_products")
    )
)


Invalid supplier rows: 0
Supplier Silver table created!


silver_rows,unique_products
111,60


In [0]:
%run ./utils_dq

In [0]:
raw = spark.read.parquet(f"{BRONZE}/supplier_price_list")
unique_rows = raw.dropDuplicates()

known = spark.table("atliq_dl.silver.products").select("product_id")

supplier_ids = unique_rows.withColumn(
    "product_id",
    F.expr("try_cast(trim(cast(product_id AS STRING)) AS BIGINT)")
)

log_dq("supplier_price_list", "exact_duplicates_removed",
       raw.count() - unique_rows.count())

log_dq("supplier_price_list", "unknown_products_excluded",
       supplier_ids.join(known, "product_id", "left_anti").count())

log_dq("supplier_price_list", "currency_format_cleaned",
       raw.filter(
           F.col("supplier_cost").cast("string").rlike(r"(?i)(₹|rs\.?|,)")
       ).count())

log_dq("supplier_price_list", "non_iso_dates_parsed",
       raw.filter(
           F.col("effective_date").isNotNull() &
           ~F.col("effective_date").cast("string").rlike(r"^\d{4}-\d{2}-\d{2}$")
       ).count())

display(
    spark.table("atliq_dl.silver.dq_log")
    .filter("table_name = 'supplier_price_list'")
    .orderBy("rule")
)

run_date,table_name,rule,rows_affected
2026-11-01,supplier_price_list,currency_format_cleaned,33
2026-10-31,supplier_price_list,currency_format_cleaned,33
2026-11-01,supplier_price_list,exact_duplicates_removed,4
2026-10-31,supplier_price_list,exact_duplicates_removed,4
2026-10-31,supplier_price_list,non_iso_dates_parsed,40
2026-11-01,supplier_price_list,non_iso_dates_parsed,40
2026-10-31,supplier_price_list,unknown_products_excluded,3
2026-11-01,supplier_price_list,unknown_products_excluded,3
